# Moon phase vs crime/accident (sklearn)
Fetch Supabase data, merge moon phases with daily outcomes, and quantify relationships via linear models.

In [68]:
import os
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from supabase import create_client

load_dotenv()
if not os.getenv("SUPABASE_URL") or not os.getenv("SUPABASE_KEY"):
    raise SystemExit("Missing SUPABASE_URL or SUPABASE_KEY in environment")

supabase = create_client(os.environ["SUPABASE_URL"], os.environ["SUPABASE_KEY"])

def fetch_table(name: str, batch_size: int = 1000, limit: int | None = None) -> pd.DataFrame:
    rows, start = [], 0
    while True:
        end = start + batch_size - 1
        resp = supabase.table(name).select("*").range(start, end).execute()
        batch = resp.data or []
        rows.extend(batch)
        if len(batch) < batch_size or (limit and len(rows) >= limit):
            break
        start += batch_size
    if limit:
        rows = rows[:limit]
    return pd.DataFrame(rows)


After fetch Function run, get all tables and store on the variables

In [69]:
crimes_df = fetch_table("chicago_crimes")
accident_df = fetch_table("chicago_accident_cleaned")
moon_df = fetch_table("cleaned_moon_data")

print(f"Crimes: {len(crimes_df)} rows")
print(f"Accidents: {len(accident_df)} rows")
print(f"Moon: {len(moon_df)} rows")

if crimes_df.empty or accident_df.empty or moon_df.empty:
    raise SystemExit("One or more tables are empty; cannot proceed")


Crimes: 251337 rows
Accidents: 110720 rows
Moon: 1826 rows


Before AI analysis some alterations and merges were performed. Firstly, grouping the Datasets rows daily incrementing the `crime_count` and `incidents_count`, while changing the `date_only` from the `chicago_crimes` and the `crash_date` fo the `chicago_accident_cleaned` to `date`.

Make sure that all dates are in datetime.

Next, merging the three datasets using `date` as the join key, ad droping any row missing the `avg_phase`, `moon_category`, `crime_count` and `incidents_count`

Lastly, copying the merged dataset in a `merged_temp` and adding the weekday and months to nexts steps 

In [84]:

crimes_daily = crimes_df.groupby("date_only")["crime_count"].sum().reset_index().rename(columns={"date_only": "date"})
acc_daily = accident_df.groupby("crash_date")["incidents_count"].sum().reset_index().rename(columns={"crash_date": "date"})

crimes_daily["date"] = pd.to_datetime(crimes_daily["date"])
acc_daily["date"] = pd.to_datetime(acc_daily["date"])
moon_df["date"] = pd.to_datetime(moon_df["date"])

merged = moon_df.merge(crimes_daily, on="date", how="left").merge(acc_daily, on="date", how="left")
merged = merged.dropna(subset=["avg_phase", "moon_category", "crime_count", "incidents_count"]).copy()

merged_temp = merged.copy()
merged_temp["weekday"] = merged_temp["date"].dt.day_name()
merged_temp["month"] = merged_temp["date"].dt.month_name()

After the dataset are ready, the AI analysis using the library `Sklean` step takes place.

The `X` matrix contains the variables used by `Sklean` to try to estimate the target vector `y`

The `drop_first=True` is adopted to avoid multicollinearity.

In [85]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score

X = pd.get_dummies(merged["moon_category"], drop_first=True)
y = merged["crime_count"].fillna(0)

model = LinearRegression()
r2 = cross_val_score(model, X, y, cv=5, scoring="r2")
model.fit(X, y)
coefs = pd.Series(model.coef_, index=X.columns).sort_values()

print(f"\n========= crime_count =========")
print(f"\n5-fold R^2: mean={r2.mean():.3f}, std={r2.std():.3f}")
print("\n weights:\n", coefs)



========= crime_count =========

5-fold R^2: mean=-0.765, std=0.802

 weights:
 Waxing Crescent   -1.799724
Full Moon          3.384789
Waxing Gibbous     3.462823
New Moon           7.640299
dtype: float64


In order to look for others insightful findings, the linear regression is also use to try to predict the daily crimes using weekdays and months as a base

In [72]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score

X = pd.get_dummies(merged_temp[["weekday"]], drop_first=True)
y = merged_temp["crime_count"].fillna(0)

model = LinearRegression()
r2 = cross_val_score(model, X, y, cv=5, scoring="r2")
model.fit(X, y)
coefs = pd.Series(model.coef_, index=X.columns).sort_values()

print(f"\n===== crime_count per weekday =====")
print(f"\n5-fold R^2: mean={r2.mean():.3f}, std={r2.std():.3f}")
print("\nWeights:\n", coefs)

X = pd.get_dummies(merged_temp[["month"]].astype("category"), drop_first=True)
y = merged_temp["crime_count"].fillna(0)

model = LinearRegression()
r2 = cross_val_score(model, X, y, cv=5, scoring="r2")
model.fit(X, y)
coefs = pd.Series(model.coef_, index=X.columns).sort_values()

print(f"\n====== crime_count per month ======")
print(f"\n5-fold R^2: mean={r2.mean():.3f}, std={r2.std():.3f}")
print("\nWeights:\n", coefs)


===== crime_count per weekday =====

5-fold R^2: mean=-0.746, std=0.804

Weights:
 weekday_Thursday    -29.333506
weekday_Tuesday     -27.114756
weekday_Wednesday   -24.118662
weekday_Sunday      -20.809339
weekday_Monday      -17.970224
weekday_Saturday     -7.031128
dtype: float64

====== crime_count per month ======

5-fold R^2: mean=-0.458, std=0.769

Weights:
 month_January     -42.234409
month_February    -39.866667
month_March       -11.195699
month_December      0.439785
month_November     17.240000
month_May          38.597849
month_October      71.875269
month_August       74.894624
month_June         80.606667
month_September    85.460000
month_July         93.881720
dtype: float64


The same procedure for `crime_counts` is applied to `incidents_count`

In [80]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score

X = pd.get_dummies(merged["moon_category"], drop_first=True)
y = merged["incidents_count"].fillna(0)

model = LinearRegression()
r2 = cross_val_score(model, X, y, cv=5, scoring="r2")
model.fit(X, y)
coefs = pd.Series(model.coef_, index=X.columns).sort_values()

print(f"\n========= incidents_count =========")
print(f"\n5-fold R^2: mean={r2.mean():.3f}, std={r2.std():.3f}")
print("\n weights:\n", coefs)


========= incidents_count =========

5-fold R^2: mean=-0.021, std=0.022

 weights:
 New Moon          -3.852545
Full Moon         -2.727174
Waxing Crescent   -1.766874
Waxing Gibbous    -0.600685
dtype: float64


In [81]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score

X = pd.get_dummies(merged_temp[["weekday"]], drop_first=True)
y = merged_temp["incidents_count"].fillna(0)

model = LinearRegression()
r2 = cross_val_score(model, X, y, cv=5, scoring="r2")
model.fit(X, y)
coefs = pd.Series(model.coef_, index=X.columns).sort_values()

print(f"\n===== incidents_count per weekday =====")
print(f"\n5-fold R^2: mean={r2.mean():.3f}, std={r2.std():.3f}")
print("\nWeights:\n", coefs)

X = pd.get_dummies(merged_temp[["month"]].astype("category"), drop_first=True)
y = merged_temp["incidents_count"].fillna(0)

model = LinearRegression()
r2 = cross_val_score(model, X, y, cv=5, scoring="r2")
model.fit(X, y)
coefs = pd.Series(model.coef_, index=X.columns).sort_values()

print(f"\n====== incidents_count per month ======")
print(f"\n5-fold R^2: mean={r2.mean():.3f}, std={r2.std():.3f}")
print("\nWeights:\n", coefs)


===== incidents_count per weekday =====

5-fold R^2: mean=0.018, std=0.028

Weights:
 weekday_Monday      -14.163120
weekday_Tuesday     -13.100620
weekday_Wednesday   -11.772495
weekday_Thursday     -8.999058
weekday_Sunday       -8.159533
weekday_Saturday     -1.669261
dtype: float64

====== incidents_count per month ======

5-fold R^2: mean=0.073, std=0.066

Weights:
 month_August       -4.979785
month_September    -4.773333
month_July         -1.734624
month_June         -1.440000
month_May          -1.405591
month_March        -0.534624
month_October       7.058925
month_November      8.206667
month_February     16.420993
month_December     18.271828
month_January      23.497634
dtype: float64


In [86]:
from sklearn.ensemble import RandomForestRegressor

X = pd.concat(
    [
        merged[["avg_phase"]],
        pd.get_dummies(merged["moon_category"], prefix="moon", dummy_na=False),
    ],
    axis=1,
)
y = merged["crime_count"].fillna(0)

model = RandomForestRegressor(
    n_estimators=300, random_state=42, min_samples_leaf=5, n_jobs=-1
)
r2 = cross_val_score(model, X, y, cv=5, scoring="r2")
model.fit(X, y)
importances = pd.Series(model.feature_importances_, index=X.columns).sort_values()

print(f"\n=== Random forest on crime_count ===")
print(f"\n5-fold R^2: mean={r2.mean():.3f}, std={r2.std():.3f}")
print("\nDrivers (importance):\n", importances)


=== Random forest on crime_count ===

5-fold R^2: mean=-0.949, std=0.788

Drivers (importance):
 moon_New Moon           0.000149
moon_Full Moon          0.000241
moon_First Quarter      0.000989
moon_Waxing Gibbous     0.001117
moon_Waxing Crescent    0.001358
avg_phase               0.996146
dtype: float64


In [75]:
from sklearn.ensemble import RandomForestRegressor

X = pd.concat(
    [
        merged[["avg_phase"]],
        pd.get_dummies(merged["moon_category"], prefix="moon", dummy_na=False),
    ],
    axis=1,
)
y = merged["incidents_count"].fillna(0)

model = RandomForestRegressor(
    n_estimators=300, random_state=42, min_samples_leaf=5, n_jobs=-1
)
r2 = cross_val_score(model, X, y, cv=5, scoring="r2")
model.fit(X, y)
importances = pd.Series(model.feature_importances_, index=X.columns).sort_values()

print(f"\n=== Random forest on incidents_count ===")
print(f"\n5-fold R^2: mean={r2.mean():.3f}, std={r2.std():.3f}")
print("\nDrivers (importance):\n", importances)


=== Random forest on incidents_count ===

5-fold R^2: mean=-0.192, std=0.046

Drivers (importance):
 moon_New Moon           0.000179
moon_Full Moon          0.000596
moon_Waxing Crescent    0.000759
moon_Waxing Gibbous     0.001148
moon_First Quarter      0.001594
avg_phase               0.995724
dtype: float64


In [76]:
from sklearn.linear_model import PoissonRegressor
from sklearn.metrics import make_scorer, mean_poisson_deviance

X = pd.concat(
    [
        merged[["avg_phase"]],
        pd.get_dummies(merged["moon_category"], prefix="moon", dummy_na=False),
    ],
    axis=1,
)
y = merged["crime_count"].fillna(0)
model = PoissonRegressor(max_iter=300)
scorer = make_scorer(mean_poisson_deviance, greater_is_better=False)
dev = -cross_val_score(model, X, y, cv=5, scoring=scorer)
model.fit(X, y)
coefs = pd.Series(model.coef_, index=X.columns).sort_values()

print(f"\n========= Poisson regression on crime_count =========")
print(f"\n5-fold mean Poisson deviance: mean={dev.mean():.3f}, std={dev.std():.3f}")
print("\nEffects:\n", coefs)


========= Poisson regression on crime_count =========

5-fold mean Poisson deviance: mean=15.735, std=6.884

Effects:
 moon_Waxing Crescent   -0.009981
moon_First Quarter     -0.003744
avg_phase              -0.000129
moon_New Moon           0.001504
moon_Waxing Gibbous     0.004835
moon_Full Moon          0.007387
dtype: float64


In [77]:
from sklearn.linear_model import PoissonRegressor
from sklearn.metrics import make_scorer, mean_poisson_deviance

X = pd.concat(
    [
        merged[["avg_phase"]],
        pd.get_dummies(merged["moon_category"], prefix="moon", dummy_na=False),
    ],
    axis=1,
)
y = merged["incidents_count"].fillna(0)
model = PoissonRegressor(max_iter=300)
scorer = make_scorer(mean_poisson_deviance, greater_is_better=False)
dev = -cross_val_score(model, X, y, cv=5, scoring=scorer)
model.fit(X, y)
coefs = pd.Series(model.coef_, index=X.columns).sort_values()

print(f"\n========== Poisson regression on incidents_count ==========")
print(f"\n5-fold mean Poisson deviance: mean={dev.mean():.3f}, std={dev.std():.3f}")
print("\nEffects:\n", coefs)



========== Poisson regression on incidents_count ==========

5-fold mean Poisson deviance: mean=6.061, std=0.251

Effects:
 moon_Full Moon         -0.038343
moon_Waxing Gibbous    -0.005176
avg_phase               0.000658
moon_New Moon           0.009706
moon_First Quarter      0.016240
moon_Waxing Crescent    0.017575
dtype: float64


In [78]:
merged_temp = merged.copy()
merged_temp["weekday"] = merged_temp["date"].dt.day_name()
merged_temp["month"] = merged_temp["date"].dt.month_name()

In [79]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score

X = pd.get_dummies(merged_temp[["weekday"]].astype("category"),  drop_first=True)
y = merged_temp["crime_count"].fillna(0)

model = LinearRegression()
r2 = cross_val_score(model, X, y, cv=5, scoring="r2")
model.fit(X, y)
coefs = pd.Series(model.coef_, index=X.columns).sort_values()

print(f"\n========= crime_count =========")
print(f"\n5-fold R^2: mean={r2.mean():.3f}, std={r2.std():.3f}")
print("\nWeights:\n", coefs)

X = pd.get_dummies(merged_temp[["month"]].astype("category"), drop_first=True)
y = merged_temp["crime_count"].fillna(0)

model = LinearRegression()
r2 = cross_val_score(model, X, y, cv=5, scoring="r2")
model.fit(X, y)
coefs = pd.Series(model.coef_, index=X.columns).sort_values()

print(f"\n========= crime_count =========")
print(f"\n5-fold R^2: mean={r2.mean():.3f}, std={r2.std():.3f}")
print("\nWeights:\n", coefs)



========= crime_count =========

5-fold R^2: mean=-0.746, std=0.804

Weights:
 weekday_Thursday    -29.333506
weekday_Tuesday     -27.114756
weekday_Wednesday   -24.118662
weekday_Sunday      -20.809339
weekday_Monday      -17.970224
weekday_Saturday     -7.031128
dtype: float64

========= crime_count =========

5-fold R^2: mean=-0.458, std=0.769

Weights:
 month_January     -42.234409
month_February    -39.866667
month_March       -11.195699
month_December      0.439785
month_November     17.240000
month_May          38.597849
month_October      71.875269
month_August       74.894624
month_June         80.606667
month_September    85.460000
month_July         93.881720
dtype: float64
